# Iterative Orchestrator

Explore the harness-controlled orchestrator that runs research as a long transaction with iterative refinement.
This orchestrator implements the AGENTS.md harness inner loop: Plan → Execute → Verify → Reflect.

## Learning Objectives
- Understand the iterative graph structure
- See how quality thresholds control iteration
- Observe how failure hints improve each pass

In [ ]:
import os
import sys
import json
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

from dotenv import load_dotenv
load_dotenv(os.path.join('..', '.env'))

print("Environment loaded")

## 1. Graph Structure

The orchestrator is a LangGraph with these nodes:

```
normalize → plan → execute → verify → observe → [iterate|finalize]
                ↑                                       │
                └───────────── iterate ─────────────────┘
```

In [ ]:
from agents.orchestrator.agent import build_graph, should_iterate
from agents.orchestrator.state import ResearchState

graph = build_graph()
print("Graph nodes:", list(graph.nodes.keys()) if hasattr(graph, 'nodes') else "(compiled graph)")
print("\nGraph structure: normalize → plan → execute → verify → observe → [iterate|finalize]")

## 2. Iteration Control Logic

The `should_iterate` function determines whether to loop or finalize:

In [ ]:
# Simulate different scenarios
scenarios = [
    {"quality_score": 4.0, "quality_threshold": 7.0, "iteration": 1, "max_iterations": 3},
    {"quality_score": 6.5, "quality_threshold": 7.0, "iteration": 2, "max_iterations": 3},
    {"quality_score": 8.0, "quality_threshold": 7.0, "iteration": 2, "max_iterations": 3},
    {"quality_score": 5.0, "quality_threshold": 7.0, "iteration": 3, "max_iterations": 3},
]

print("Iteration control decisions:")
print(f"{'Score':>6} {'Threshold':>10} {'Iteration':>10} {'Max':>4} {'Decision':>10}")
print("-" * 50)
for s in scenarios:
    decision = should_iterate(s)
    print(f"{s['quality_score']:>6.1f} {s['quality_threshold']:>10.1f} {s['iteration']:>10} {s['max_iterations']:>4} {decision:>10}")

## 3. Running the Orchestrator

Let's run the full orchestrator graph on a sample query:

In [ ]:
import uuid

# Configure initial state
initial_state = {
    "session_id": str(uuid.uuid4())[:12],
    "query": "What are the key benefits of using vector databases for RAG applications?",
    "file_path": "",
    "has_document": False,
    "iteration": 0,
    "max_iterations": 2,  # Limit for demo
    "quality_threshold": 7.0,
    "research_plan": [],
    "accumulated_context": [],
    "current_draft": "",
    "verification_result": {},
    "verification_history": [],
    "quality_score": 0.0,
    "total_tokens": 0,
    "total_cost": 0.0,
    "failure_hints": "",
    "status": "normalizing",
    "final_output": "",
    "error": "",
}

print(f"Session: {initial_state['session_id']}")
print(f"Query: {initial_state['query']}")
print(f"Max iterations: {initial_state['max_iterations']}")
print(f"Quality threshold: {initial_state['quality_threshold']}")

In [ ]:
# Run the graph (this will iterate until quality threshold or max iterations)
import asyncio

result = asyncio.get_event_loop().run_until_complete(graph.ainvoke(initial_state))

print(f"\n{'='*60}")
print(f"Session completed!")
print(f"  Status: {result.get('status')}")
print(f"  Iterations: {result.get('iteration')}")
print(f"  Final quality score: {result.get('quality_score', 0):.1f}")
print(f"  Total tokens: {result.get('total_tokens', 0)}")
print(f"  Context items: {len(result.get('accumulated_context', []))}")
print(f"\nVerification history:")
for v in result.get('verification_history', []):
    print(f"  Iter {v.get('iteration')}: score={v.get('score', 0):.1f}, passed={v.get('passed')}")

In [ ]:
# View the final output
print("Final Output:")
print("=" * 60)
print(result.get('final_output', 'No output')[:2000])

## Summary

The iterative orchestrator:
- Runs research as a **long transaction** with multiple passes
- Uses **quality thresholds** to determine when output is sufficient
- Accumulates **failure hints** that improve each iteration
- Tracks **metrics** (tokens, cost, latency) across iterations

This is the core of the AGENTS.md harness — the iterative inner loop that drives research quality upward.